# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

[Dataset Croissant Schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print Dataset Name and Description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs. All entities are referenced using their `@id` fields as per Croissant schema.

In [ ]:
# Get all record sets by @id
record_sets_dict = dataset.metadata.recordSet
if record_sets_dict is None or len(record_sets_dict) == 0:
    # For some schemas (depending on mlcroissant parsing), .recordSet may be a list or missing
    # fallback: find record sets from the top-level metadata JSONLD
    import requests
    import json
    resp = requests.get(croissant_url)
    croissant_json = resp.json()
    # Collect record sets based on '@type' == 'RecordSet' or cr:RecordSet
    record_sets = []
    for obj in croissant_json.get('@graph', croissant_json if isinstance(croissant_json, list) else []):
        if ('@type' in obj and (obj['@type'] == 'RecordSet' or obj['@type'] == 'cr:RecordSet')):
            record_sets.append(obj)
    record_sets_ids = [rs['@id'] for rs in record_sets]
else:
    if isinstance(record_sets_dict, dict):
        record_sets_ids = list(record_sets_dict.keys())
    elif isinstance(record_sets_dict, list):
        record_sets_ids = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets_dict]
    else:
        record_sets_ids = []

print("Available Record Sets (@id):")
for rsid in record_sets_ids:
    print(f"- {rsid}")

# If record set metadata is accessible, show its fields and columns:
fields_dict = getattr(dataset.metadata, 'field', None)
if fields_dict is None:
    # fallback - find from schema
    fields_dict = {}
    if 'record_sets' in locals():
        for rs in record_sets:
            if 'field' in rs:
                for fld in rs['field'] if isinstance(rs['field'], list) else [rs['field']]:
                    if isinstance(fld, dict) and '@id' in fld:
                        fields_dict[fld['@id']] = fld
                    elif isinstance(fld, str):
                        fields_dict[fld] = None

print("Fields (@id):")
for fid in fields_dict:
    print(f"- {fid}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Record sets and fields are referenced by their `@id`s. Below, the notebook dynamically extracts all record sets and loads their records.

In [ ]:
# Extract data from each record set
dataframes = {}

# Loop through all discovered record set IDs
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {record_set_id} with {len(records)} records.")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
        continue

# Show head of first available record set
if len(dataframes) > 0:
    first_rs = list(dataframes.keys())[0]
    df = dataframes[first_rs]
    print(f"Sample records from {first_rs}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Use the `@id` for fields/columns.
- Demonstrate filtering, normalization, and grouping.


In [ ]:
# For demonstration, use the main survey record set (you may adjust record_set_id as needed):
record_set_id = first_rs # Change if you want a specific record set
df = dataframes[record_set_id]

# Choose numeric fields for analysis (example: 'phq_9_score', 'gad_7_score', 'psq_score')
# These are typically referenced by their @id, but we will try columns found in df
possible_numeric_fields = [col for col in df.columns if 'score' in col or col.startswith('phq_9') or col.startswith('gad_7') or col.startswith('psq')]
print("Possible numeric fields for analysis:", possible_numeric_fields)

# Use first found numeric field
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (example: 'gender', which may be referenced by @id)
    group_field_candidates = [col for col in df.columns if col in [
        'gender', 'village', 'age_group', 'level_of_education', 'marital_status']]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we visualize the distribution of the selected numeric field and its relationship to a categorical group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields and group_field_candidates:
    # Distribution plot for the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot: numeric field by group field
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the FAIR² Mental Health Survey Dataset using `mlcroissant` referencing all entities by their `@id`.

- **Data loaded**: We accessed all record sets and fields using their unique `@id`s.
- **Overview**: Listed record sets and fields for the dataset, enabling transparent exploration.
- **Extraction & EDA**: Showed how to filter, normalize, and group by key demographic attributes.
- **Visualization**: Displayed distributions and relationships between mental health scores and demographic variables.

For further exploration, use the identified `@id`s for any dataset entity and apply advanced analysis with `mlcroissant` and pandas.